# Bab 5: Learning Multiple Weights at a Time — Generalizing Gradient Descent

Notebook ini merangkum Bab 5 buku *Grokking Deep Learning* karya Andrew W. Trask. Bab sebelumnya membahas gradient descent untuk satu weight saja; bab ini memperluasnya ke network dengan banyak input, banyak output, dan kombinasi keduanya. Di akhir bab, konsep ini mulai dikaitkan dengan dataset nyata pertama di buku ini: MNIST (gambar tulisan tangan digit).

## 1. Gradient Descent dengan Banyak Input

Prinsipnya sama seperti satu weight, hanya saja sekarang kita punya tiga weight yang masing-masing butuh `weight_delta`-nya sendiri. Karena ketiga weight berbagi output node yang sama, mereka juga berbagi `delta` yang sama, tapi `weight_delta` tetap berbeda-beda karena nilai input masing-masing weight berbeda.

In [1]:
def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, -0.1]

def neural_network(input, weights):
    pred = w_sum(input, weights)
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

win_or_lose_binary = [1, 1, 0, 1]
true = win_or_lose_binary[0]
input = [toes[0], wlrec[0], nfans[0]]

# PREDICT + COMPARE
pred = neural_network(input, weights)
error = (pred - true) ** 2
delta = pred - true

# LEARN: hitung weight_delta untuk tiap weight
def ele_mul(number, vector):
    output = [0, 0, 0]
    assert len(output) == len(vector)
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weight_deltas = ele_mul(delta, input)

# LEARN: update tiap weight
alpha = 0.01
for i in range(len(weights)):
    weights[i] -= alpha * weight_deltas[i]

print("Weights:" + str(weights))
print("Weight Deltas:" + str(weight_deltas))

Weights:[0.1119, 0.20091, -0.09832]
Weight Deltas:[-1.189999999999999, -0.09099999999999994, -0.16799999999999987]


### Bagaimana Satu Delta Berubah Jadi Tiga Weight Delta?

Perlu dibedakan dua konsep ini:

- **delta**: ukuran seberapa besar kita ingin nilai sebuah node berubah, dihitung langsung dari `pred - true`. Delta ini milik output node, jadi sama untuk semua weight yang terhubung ke node itu.
- **weight_delta**: estimasi arah dan jumlah perubahan suatu weight untuk mengurangi node delta, dihasilkan dari derivatif. Weight_delta dihitung dengan mengalikan `delta` dengan **input milik weight tersebut**.

Karena tiap weight punya input yang unik tapi delta yang sama, masing-masing weight_delta otomatis menjadi "versi delta yang dimodifikasi sesuai input"-nya sendiri, lewat mekanisme scaling, negative reversal, dan stopping yang sudah dibahas di bab sebelumnya.

## 2. Mengamati Beberapa Langkah Pembelajaran

Mari jalankan beberapa iterasi dan amati bagaimana tiap weight bergerak menuju titik error minimumnya masing-masing.

In [2]:
def neural_network(input, weights):
    out = 0
    for i in range(len(input)):
        out += (input[i] * weights[i])
    return out

def ele_mul(scalar, vector):
    out = [0, 0, 0]
    for i in range(len(out)):
        out[i] = vector[i] * scalar
    return out

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]
win_or_lose_binary = [1, 1, 0, 1]
true = win_or_lose_binary[0]
alpha = 0.01
weights = [0.1, 0.2, -0.1]
input = [toes[0], wlrec[0], nfans[0]]

for iter in range(3):
    pred = neural_network(input, weights)
    error = (pred - true) ** 2
    delta = pred - true
    weight_deltas = ele_mul(delta, input)

    print("Iteration:" + str(iter + 1))
    print("Pred:" + str(pred))
    print("Error:" + str(error))
    print("Delta:" + str(delta))
    print("Weights:" + str(weights))
    print("Weight_Deltas:")
    print(str(weight_deltas))
    print()

    for i in range(len(weights)):
        weights[i] -= alpha * weight_deltas[i]

Iteration:1
Pred:0.8600000000000001
Error:0.01959999999999997
Delta:-0.1399999999999999
Weights:[0.1, 0.2, -0.1]
Weight_Deltas:
[-1.189999999999999, -0.09099999999999994, -0.16799999999999987]

Iteration:2
Pred:0.9637574999999999
Error:0.0013135188062500048
Delta:-0.036242500000000066
Weights:[0.1119, 0.20091, -0.09832]
Weight_Deltas:
[-0.30806125000000056, -0.023557625000000044, -0.04349100000000008]

Iteration:3
Pred:0.9906177228125002
Error:8.802712522307997e-05
Delta:-0.009382277187499843
Weights:[0.11498061250000001, 0.20114557625, -0.09788509000000001]
Weight_Deltas:
[-0.07974935609374867, -0.006098480171874899, -0.011258732624999811]



Ada catatan penting dari hasil eksekusi ini: sebagian besar perubahan weight terjadi pada weight dengan input terbesar (`toes`, karena nilainya 8.5, jauh lebih besar dari `wlrec` atau `nfans`), karena input yang besar membuat slope-nya jauh lebih tajam. Ini bukan selalu hal yang menguntungkan, dan ada subbidang bernama **normalization** yang membantu mendorong proses belajar lebih merata di semua weight, terlepas dari karakteristik skala datasetnya. Perbedaan slope yang signifikan ini juga memaksa `alpha` diset lebih rendah dari yang diinginkan (0.01, bukan 0.1), karena kalau dinaikkan, weight dengan input besar bisa menyebabkan divergence.

## 3. Eksperimen: Membekukan Satu Weight

Eksperimen menarik untuk memahami bagaimana weight-weight saling berkaitan: kita latih network seperti biasa, tapi weight pertama (`a`, terkait `toes`) tidak pernah diupdate (`weight_deltas[0]` dipaksa jadi 0). Network hanya belajar lewat weight `b` dan `c`.

In [3]:
def neural_network(input, weights):
    out = 0
    for i in range(len(input)):
        out += (input[i] * weights[i])
    return out

def ele_mul(scalar, vector):
    out = [0, 0, 0]
    for i in range(len(out)):
        out[i] = vector[i] * scalar
    return out

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]
win_or_lose_binary = [1, 1, 0, 1]
true = win_or_lose_binary[0]
alpha = 0.3
weights = [0.1, 0.2, -0.1]
input = [toes[0], wlrec[0], nfans[0]]

for iter in range(3):
    pred = neural_network(input, weights)
    error = (pred - true) ** 2
    delta = pred - true
    weight_deltas = ele_mul(delta, input)
    weight_deltas[0] = 0  # weight 'a' dibekukan, tidak pernah diupdate

    print("Iteration:" + str(iter + 1))
    print("Pred:" + str(pred))
    print("Error:" + str(error))
    print("Weights:" + str(weights))
    print("Weight_Deltas:")
    print(str(weight_deltas))
    print()

    for i in range(len(weights)):
        weights[i] -= alpha * weight_deltas[i]

Iteration:1
Pred:0.8600000000000001
Error:0.01959999999999997
Weights:[0.1, 0.2, -0.1]
Weight_Deltas:
[0, -0.09099999999999994, -0.16799999999999987]

Iteration:2
Pred:0.9382250000000001
Error:0.003816150624999989
Weights:[0.1, 0.2273, -0.04960000000000005]
Weight_Deltas:
[0, -0.040153749999999946, -0.07412999999999989]

Iteration:3
Pred:0.97274178125
Error:0.000743010489422852
Weights:[0.1, 0.239346125, -0.02736100000000008]
Weight_Deltas:
[0, -0.017717842187500006, -0.032709862500000006]



Hasil yang menarik: error tetap bisa turun ke 0 meskipun weight `a` tidak pernah bergerak. Kenapa begitu? Karena ketiga weight berbagi error yang sama, jadi ketika salah satu weight menemukan "dasar mangkuk"-nya, semua weight (termasuk yang dibekukan, secara pasif) ikut berada di titik error minimum yang sama.

Ini mengungkap pelajaran penting (dan cukup berbahaya) tentang neural network: kalau network kebetulan sudah berhasil memprediksi dengan akurat di data training tanpa memakai weight `a` sama sekali, maka weight `a` **tidak akan pernah** belajar untuk ikut berkontribusi, meskipun secara teori `a` adalah prediktor yang kuat. Begitu error mencapai 0, `weight_delta` juga otomatis 0, jadi tidak ada lagi dorongan untuk mengubah weight apa pun, termasuk weight yang sebenarnya informatif tapi "keburu" tidak terpakai.

Secara konseptual, grafik error untuk tiap weight sebenarnya adalah potongan 2D dari bentuk 4-dimensi: tiga dimensi untuk nilai tiga weight, dan dimensi keempat untuk error. Bentuk ini disebut **error plane**, dan kelengkungannya ditentukan sepenuhnya oleh data training.

## 4. Gradient Descent dengan Banyak Output

Sebaliknya, kita juga bisa punya satu input yang menghasilkan beberapa prediksi sekaligus (misalnya memprediksi apakah pemain terluka, menang, atau sedih, semuanya hanya dari win/loss record). Caranya: hitung delta untuk masing-masing output seperti biasa, lalu kalikan semuanya dengan input yang sama (karena hanya ada satu input) untuk mendapatkan weight_delta tiap weight.

In [4]:
def neural_network(input, weights):
    pred = ele_mul(input, weights)
    return pred

def ele_mul(number, vector):
    output = [0, 0, 0]
    assert len(output) == len(vector)
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

def scalar_ele_mul(number, vector):
    output = [0, 0, 0]
    assert len(output) == len(vector)
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weights = [0.3, 0.2, 0.9]

wlrec = [0.65, 1.0, 1.0, 0.9]
hurt = [0.1, 0.0, 0.0, 0.1]
won = [1, 1, 0, 1]
sad = [0.1, 0.0, 0.1, 0.2]

input = wlrec[0]
true = [hurt[0], won[0], sad[0]]

# PREDICT
pred = neural_network(input, weights)

# COMPARE
error = [0, 0, 0]
delta = [0, 0, 0]
for i in range(len(true)):
    error[i] = (pred[i] - true[i]) ** 2
    delta[i] = pred[i] - true[i]

# LEARN
weight_deltas = scalar_ele_mul(input, weights)
alpha = 0.1
for i in range(len(weights)):
    weights[i] -= (weight_deltas[i] * alpha)

print("Weights:" + str(weights))
print("Weight Deltas:" + str(weight_deltas))

Weights:[0.28049999999999997, 0.187, 0.8415]
Weight Deltas:[0.195, 0.13, 0.5850000000000001]


Pada dasarnya ini adalah mekanisme **stochastic gradient descent** yang sama persis, hanya diterapkan secara konsisten ke arsitektur yang berbeda-beda: satu input banyak output, banyak input satu output, atau nanti kita lihat juga banyak input dan banyak output sekaligus.

## 5. Gradient Descent dengan Banyak Input dan Banyak Output

Kombinasi kedua kasus di atas: beberapa input, beberapa output, dan tiap pasangan input-output punya weight sendiri (membentuk matrix weight). `weight_delta` untuk seluruh matrix dihitung lewat **outer product** antara vektor input dan vektor delta output.

In [27]:
# toes  % win  # fans
weights = [[0.1, 0.1, -0.3],  # hurt?
           [0.1, 0.2, 0.0],   # win?
           [0.0, 1.3, 0.1]]   # sad?

def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def vect_mat_mul(vect, matrix):
    assert len(vect) == len(matrix)
    output = [0, 0, 0]
    for i in range(len(vect)):
        output[i] = w_sum(vect, matrix[i])
    return output

def neural_network(input, weights):
    pred = vect_mat_mul(input, weights)
    return pred

def zeros_matrix(a, b):
    return [[0 for _ in range(b)] for _ in range(a)]

def outer_prod(vec_a, vec_b):
    out = zeros_matrix(len(vec_a), len(vec_b))
    for i in range(len(vec_a)):
        for j in range(len(vec_b)):
            out[i][j] = vec_a[i] * vec_b[j]
    return out

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]
hurt = [0.1, 0.0, 0.0, 0.1]
won = [1, 1, 0, 1]
sad = [0.1, 0.0, 0.1, 0.2]

alpha = 0.01
input = [toes[0], wlrec[0], nfans[0]]
true = [hurt[0], won[0], sad[0]]

# PREDICT
pred = neural_network(input, weights)

# COMPARE
error = [0, 0, 0]
delta = [0, 0, 0]
for i in range(len(true)):
    error[i] = (pred[i] - true[i]) ** 2
    delta[i] = pred[i] - true[i]

# LEARN: weight_deltas sekarang berbentuk matrix, lewat outer product input dan delta
weight_deltas = outer_prod(input, delta)

for i in range(len(weights)):
    for j in range(len(weights[0])):
        weights[i][j] -= alpha * weight_deltas[i][j]

print("Weights:" + str(weights))

Weights:[[0.061325, 0.1017, -0.373525], [0.0970425, 0.20013, -0.005622500000000002], [-0.0054600000000000004, 1.30024, 0.08962]]


Pola yang berulang di sini jelas sekali: setiap kali kita menambah kompleksitas arsitektur (banyak input, banyak output, atau keduanya), mekanisme gradient descent-nya tetap sama (predict, compare, learn), hanya operasi matematikanya (weighted sum, elementwise multiplication, vector-matrix multiplication, atau outer product) yang berubah menyesuaikan bentuk network-nya.

## 6. Dataset Pertama yang Nyata: MNIST

Bab ini ditutup dengan memperkenalkan dataset nyata pertama di buku: **MNIST**, kumpulan gambar tulisan tangan digit (0–9) yang ditulis oleh siswa SMA dan pegawai Biro Sensus AS. Dataset ini sudah dipakai berpuluh tahun sebagai benchmark untuk melatih network mengenali tulisan tangan manusia.

Spesifikasi dataset:

- Setiap gambar berukuran 28×28 piksel, yang kemudian **diratakan** (flatten) menjadi vektor 1×784, dengan menggabungkan tiap baris piksel secara berurutan.
- Network butuh **784 input node** (satu untuk setiap piksel) dan **10 output node** (satu untuk setiap kemungkinan digit 0–9).
- Network ini pada dasarnya adalah perluasan langsung dari network banyak-input-banyak-output yang sudah dibahas, hanya jumlah input dan output-nya jauh lebih besar.

Kalau network ini bisa memprediksi dengan sempurna, untuk gambar digit 2 misalnya, network akan menghasilkan 1.0 di posisi output ketiga (indeks untuk angka 2) dan 0 di semua posisi lain.

## 7. Memvisualisasikan Nilai Weight

Praktik yang menarik dan intuitif dalam riset neural network (khususnya untuk classifier gambar) adalah memvisualisasikan weight sebagai gambar. Setiap output node (misalnya node "2?") punya 784 weight input, masing-masing memetakan hubungan antara satu piksel dengan angka 2.

- Weight yang **tinggi** berarti model percaya ada korelasi tinggi antara piksel itu dengan angka 2.
- Weight yang **sangat rendah (negatif)** berarti model percaya korelasinya rendah, bahkan mungkin negatif.

Kalau kumpulan weight ini digambar ulang dalam bentuk gambar berukuran sama dengan gambar input aslinya, kita bisa melihat pola samar bentuk angka 2 atau 1 (area terang = weight tinggi, area gelap = weight negatif). Ini menunjukkan bahwa network memang "mengetahui" secara kasar bentuk visual dari tiap digit.

## 8. Mengapa Ini Bekerja: Kembali ke Dot Product

Penjelasan di balik visualisasi tersebut kembali ke konsep dot product yang sudah dibahas di Bab 3: dot product adalah ukuran kemiripan antara dua vektor. Kalau vektor weight untuk angka 2 mirip dengan vektor input gambar yang memang berisi tulisan angka 2, hasil dot product-nya (skor prediksi) akan tinggi. Sebaliknya, kalau tidak mirip, skornya akan rendah.

```
a = [0, 1, 0, 1]
b = [1, 0, 1, 0]   -> dot(a, b) = 0  (tidak ada kemiripan)

c = [0, 1, 1, 0]
d = [.5, 0, .5, 0] -> dot(c, d) lebih tinggi karena ada overlap nilai positif
```

Inilah sebabnya skor prediksi untuk gambar digit 2 bisa jauh lebih tinggi (misalnya 0.98) di output node "2?" dibanding output node lainnya (misalnya 0.01): weight node "2?" memang sudah "belajar" menyerupai pola piksel khas angka 2.

## Poin-poin Penting Bab 5

- Gradient descent bisa digeneralisasi ke network dengan banyak input, banyak output, atau kombinasi keduanya, tanpa mengubah prinsip dasarnya: predict, compare, learn.
- `delta` adalah milik output node (seberapa jauh node itu meleset), sementara `weight_delta` adalah turunan `delta` yang sudah disesuaikan dengan input milik masing-masing weight.
- Untuk banyak input: `weight_deltas` dihitung lewat elementwise multiplication antara `delta` (sama untuk semua weight) dan vektor input (unik tiap weight).
- Untuk banyak output: `weight_deltas` dihitung lewat elementwise multiplication antara input tunggal (sama untuk semua weight) dan vektor delta (unik tiap output).
- Untuk banyak input dan output: `weight_deltas` berbentuk matrix, dihitung lewat **outer product** antara vektor input dan vektor delta.
- Input dengan skala jauh lebih besar dari input lain akan mendominasi proses belajar (slope-nya lebih tajam), karena itu normalisasi data menjadi penting.
- Membekukan satu weight (tidak pernah diupdate) tetap bisa membuat error turun ke 0 karena error-nya bersama (shared), tapi ini mengungkap risiko: weight yang sebenarnya informatif bisa jadi tidak pernah dipelajari kalau network sudah "kebetulan" akurat tanpa weight itu.
- Dataset MNIST (784 piksel input, 10 output digit) diperkenalkan sebagai aplikasi nyata pertama, dan weight yang dipelajari bisa divisualisasikan sebagai gambar yang menyerupai bentuk digit terkait, karena pada dasarnya prediksi adalah pengukuran kemiripan (dot product) antara weight dan input.

Referensi: Trask, A. W. (2019). *Grokking Deep Learning*, Bab 5 "Learning Multiple Weights at a Time: Generalizing Gradient Descent". Manning Publications.